In [ ]:
#| default_exp features.ocf_ontarget

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class OCFOntargetEvaluator(FeatureEvaluator):
    """Extracts on-target OCF metrics per tissue."""
    
    name = "OCFOntarget"
    source_file = ".OCF.ontarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "tissue" in cols:
                for _, row in df.iterrows():
                    t = str(row["tissue"]).replace(" ", "_")
                    if "OCF" in cols and pd.notna(row["OCF"]): extracted[f"{t}_OCF"] = float(row["OCF"])
                    if "ocf_z" in cols and pd.notna(row["ocf_z"]): extracted[f"{t}_ocf_z"] = float(row["ocf_z"])
    
            return extracted
        except Exception as e:
            log.warning("ocf_ontarget_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = OCFOntargetEvaluator()
df = pd.DataFrame([{"tissue":"Breast","OCF":-13.0,"ocf_z":1.2}])
res = evaluator.extract(df)
assert res["Breast_OCF"] == -13.0
